## Imports and Configuration

In [ ]:
# Core
import pandas as pd
import numpy as np

# Dates
from datetime import datetime, timedelta

# Optional but VERY helpful later for holidays
import holidays

# Display tweaks (makes debugging way nicer)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# File path
DATA_FILE = "NewResults.csv"  # <-- change this

# Toggle for test mode
TEST_MODE = False
TEST_METERS = 5   # number of meters to include in test
TEST_DAYS = 100     # number of days per meter

## Load Data

In [ ]:
df = pd.read_csv(DATA_FILE)

print("Shape:", df.shape)
df.head()

In [ ]:
# SCRATCH CHECKING
print("Unique multipliers:", df["Multiplier"].nunique())
print(df["Multiplier"].unique())
#Unique cycles
print("Unique cycles:", df["CISCycleIdentifier"].nunique())
print(df["CISCycleIdentifier"].unique())

## Basic Cleaning and Typing

In [ ]:
print("Before cleaning:", df.shape)

# Convert datetime
df["ReadLogDate"] = pd.to_datetime(df["ReadLogDate"], errors="coerce")
df["ReadDate"] = pd.to_datetime(df["ReadDate"], errors="coerce")

# Numeric safety
df["ReadValue"] = pd.to_numeric(df["ReadValue"], errors="coerce")
df["Multiplier"] = pd.to_numeric(df["Multiplier"], errors="coerce")

# Drop obvious garbage rows
df = df.dropna(subset=["ReadDate", "ReadValue"])

# Sort for sanity
df = df.sort_values(["MeterIdentifier", "ReadDate"]).reset_index(drop=True)

print("After cleaning:", df.shape)

## Sanity Check

In [ ]:
print("Unique meters:", df["MeterIdentifier"].nunique())
print("Date range:", df["ReadDate"].min(), "→", df["ReadDate"].max())

# Check interval spacing (sample one meter)
sample_meter = df["MeterIdentifier"].iloc[0]
sample = df[df["MeterIdentifier"] == sample_meter].copy()

sample["delta"] = sample["ReadDate"].diff()
sample["delta_minutes"] = sample["delta"] / pd.Timedelta(minutes=1)
print(sample["delta"].value_counts().head())
print(sample["delta_minutes"].value_counts().head())

# Create Run Subset

In [ ]:
if TEST_MODE:
    # Pick a few meters
    meters = df["MeterIdentifier"].dropna().unique()[:TEST_METERS]
    test_df = df[df["MeterIdentifier"].isin(meters)].copy()

    # Limit to first N days per meter
    test_df["date"] = test_df["ReadDate"].dt.date
    
    test_df = (
        test_df.sort_values("ReadDate")
        .groupby("MeterIdentifier")
        .apply(lambda x: x[x["date"] <= (x["date"].min() + timedelta(days=TEST_DAYS))])
        .reset_index(drop=True)
    )

    df_test = test_df.drop(columns=["date"])

    print("Test shape:", df_test.shape)
else:
    df_test = df.copy()

## Base Time Columns

In [ ]:
df_test["hour"] = df_test["ReadDate"].dt.hour
df_test["minute"] = df_test["ReadDate"].dt.minute
df_test["date"] = df_test["ReadDate"].dt.date
df_test["month"] = df_test["ReadDate"].dt.month
df_test["weekday"] = df_test["ReadDate"].dt.weekday  # 0=Mon, 6=Sun

# Weekend flag
df_test["is_weekend"] = df_test["weekday"] >= 5

## Flag Meter Reads Missing Multipliers and Fill with 1s

In [ ]:
df_test["missing_multiplier"] = df_test["Multiplier"].isna()
df_test["Multiplier"] = df_test["Multiplier"].fillna(1)

## Calc Multiplied Usage

In [ ]:
df_test["usage_kwh_calc"] = df_test["ReadValue"] * df_test["Multiplier"]
print(df_test.shape)

## Set Up Holidays

In [ ]:
# US holidays (you can customize later)
us_holidays = holidays.US()

def get_holiday_set(years):
    holiday_dates = set()
    for y in years:
        for date in holidays.US(years=y).keys():
            holiday_dates.add(date)
    return holiday_dates

years_in_data = df_test["ReadLogDate"].dt.year.unique()
holiday_dates = get_holiday_set(years_in_data)

len(holiday_dates)

In [ ]:
df_test.head(10)


# Core Functions

In [ ]:
# ON/OFF-PEAK CLASSIFICATION
def is_on_peak(row):
    dt = row["ReadDate"]
    
    hour = dt.hour
    month = dt.month
    day = dt.day
    weekday = dt.weekday()  # 0=Mon, 6=Sun
    
    # Weekend → OFF PEAK
    if weekday >= 5:
        return False
    
    # Good Friday (2026-04-03) → OFF PEAK
    if dt.date() == datetime(2026, 4, 3).date():
        return False
    
    # Jan–Mar → 6–9 AM
    if month in [1, 2, 3]:
        return 6 <= hour < 9
    
    # Apr 1–15 → 6–9 AM AND 1–6 PM
    if month == 4 and day <= 15:
        return (6 <= hour < 9) or (13 <= hour < 18)

    # Apr 16–30 → 1–6 PM
    if month == 4 and day >= 16:
        return (13 <= hour < 18)

    # May–Sep → 1–6 PM
    if month in [5, 6, 7, 8, 9]:
        return (13 <= hour < 18)
        return False

In [ ]:
# Test if Good Friday (Friday, normally has on-peak) is wholly off-peak

df_test["on_peak"] = df_test.apply(is_on_peak, axis=1)

df_test[
    df_test["ReadDate"].dt.date == datetime(2026, 4, 3).date()
]["on_peak"].value_counts()


## Generate Summary

In [ ]:
summary = df_test.groupby(["MeterIdentifier", "on_peak", "Multiplier", "missing_multiplier"]).agg(usage_kwh_calc_sum=("usage_kwh_calc", "sum"), usage_kwh_raw_sum=("ReadValue", "sum")).reset_index()
print(summary)

In [ ]:
# --- TRANSFORM LONG → WIDE (ONE ROW PER METER) ---

# 1. Define all the metadata fields you want to keep
# This includes the cycle ID, account info, and multiplier flags
meta_fields = [
    "AccountNumber",
    "AccountSubNumber",
    "AccountRate",
    "CISCycleIdentifier",  # The one you requested
    "Multiplier",
    "missing_multiplier"
]

# 2. Extract these fields from df_test (one row per meter)
# We use 'first' because these values should be identical across all rows for a meter
meter_meta = df_test.groupby("MeterIdentifier").agg({
    col: "first" for col in meta_fields if col in df_test.columns
}).reset_index()

# 3. Pivot the usage data so on-peak/off-peak are side-by-side
pivot_df = summary.pivot_table(
    index="MeterIdentifier",
    columns="on_peak",
    values=["usage_kwh_calc_sum", "usage_kwh_raw_sum"],
    aggfunc="sum"
)

# 4. Flatten columns (e.g., usage_kwh_calc_sum_on_peak)
pivot_df.columns = [
    f"{metric}_{'on_peak' if peak else 'off_peak'}"
    for metric, peak in pivot_df.columns
]

# 5. Merge the metadata and the usage data together
final_df = pivot_df.reset_index().merge(meter_meta, on="MeterIdentifier", how="left")

# 6. Fill missing usage with 0 and organize columns for the final CSV
final_df = final_df.fillna(0)

# Organize columns: ID first, then Meta, then Usage totals
usage_cols = [
    "usage_kwh_calc_sum_off_peak", "usage_kwh_calc_sum_on_peak",
    "usage_kwh_raw_sum_off_peak", "usage_kwh_raw_sum_on_peak"
]
column_order = ["MeterIdentifier"] + [f for f in meta_fields if f in final_df.columns] + usage_cols
final_df = final_df[column_order]

# Save the truly consolidated file
final_df.to_csv("summary_consolidated.csv", index=False)

print(f"Consolidated summary created with {len(final_df)} meters.")
final_df.head()


In [ ]:
# Create the summary table matching the SQL output format
debug_df = final_df.copy()
# Rename columns to match SQL names
debug_df = debug_df.rename(columns={
    "Multiplier": "MultiplierUsed",
    "usage_kwh_raw_sum_on_peak": "OnPeak_kWh",
    "usage_kwh_raw_sum_off_peak": "OffPeak_kWh",
    "usage_kwh_calc_sum_on_peak": "OnPeak_kWh_xMultiplier",
    "usage_kwh_calc_sum_off_peak": "OffPeak_kWh_xMultiplier"
})
# Calculate totals
debug_df["Total_kWh"] = debug_df["OnPeak_kWh"] + debug_df["OffPeak_kWh"]
debug_df["Total_kWh_xMultiplier"] = debug_df["OnPeak_kWh_xMultiplier"] + debug_df["OffPeak_kWh_xMultiplier"]
# Reorder columns to match the request
debug_cols = [
    "MeterIdentifier",
    "MultiplierUsed",
    "OnPeak_kWh",
    "OffPeak_kWh",
    "OnPeak_kWh_xMultiplier",
    "OffPeak_kWh_xMultiplier",
    "Total_kWh",
    "Total_kWh_xMultiplier"
]
debug_summary = debug_df[debug_cols]
# Display the result
pd.set_option('display.max_rows', None)  # Show all meters
debug_summary.sort_values("MeterIdentifier")

In [ ]:
thomas_df = pd.read_csv("thomas_sql_results.csv")
thomas_df.rename(columns={"Total_kWh": "Thomas_Total_kWh", "Total_kWh_xMultiplier": "Thomas_Total_kWh_xMultiplier"}, inplace=True)
thomas_df.head()

In [ ]:
merged_df = debug_df.merge(
    thomas_df[["MeterIdentifier", "Thomas_Total_kWh", "Thomas_Total_kWh_xMultiplier"]],
    on="MeterIdentifier",
    how="left"
)

pd.set_option('display.max_rows', None)  # Show all meters
merged_df.sort_values("MeterIdentifier")


In [ ]:
final_df.to_csv("summary_consolidated.csv", index=False)